# Phase 11 — Combien de temps ça a duré

## Objectifs

- Construire une durée exploitable en secondes pour le plus grand nombre de relevés.
- Exploiter `duration_hours_min` lorsque `duration_seconds` est absente ou incohérente.
- Ne supprimer aucune ligne.
- Compter les durées restant inutilisables, les contradictions et les durées de plus d'une journée.
- Examiner les trois durées les plus longues et prendre une décision explicite.


## 1. Imports

In [1]:
from pathlib import Path
import csv
import re

import numpy as np
import pandas as pd


## 2. Chemins et colonnes

In [2]:
DATA_PATH = Path("../data/releves_klaxo3.csv")
OUTPUT_DIR = Path("../outputs")
PHASE11_DIR = OUTPUT_DIR / "phase_11_durees"
PHASE11_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_hours_min", "comments",
    "date_posted", "latitude", "longitude",
]


## 3. Chargement robuste

In [3]:
lignes_valides = []
lignes_problemes = []

with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)
    for numero_ligne, row in enumerate(reader, start=1):
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
        else:
            lignes_problemes.append({
                "numero_ligne": numero_ligne,
                "nb_champs": len(row),
                "contenu": row,
            })

df = pd.DataFrame(lignes_valides, columns=COLUMNS)
print(f"Lignes chargées : {len(df)}")


Lignes chargées : 88679


## 4. Préparation des deux colonnes de durée

La colonne `duration_seconds` est convertie en nombre. La colonne `duration_hours_min` est conservée sous forme de texte normalisé afin d'en extraire une durée lorsque cela est possible.

In [4]:
df["duration_seconds_original"] = df["duration_seconds"]
df["duration_seconds"] = pd.to_numeric(
    df["duration_seconds"],
    errors="coerce",
)

df["duration_hours_min_clean"] = (
    df["duration_hours_min"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)


## 5. Fonction de conversion du texte vers des secondes

Cette fonction couvre les formes les plus fréquentes : secondes, minutes, heures, jours, plages de valeurs et expressions simples comme `half hour`. Les formats non interprétables retournent `NaN`.


In [5]:
def parse_duration_to_seconds(value):
    texte = str(value).lower().strip()

    if texte in {"", "nan", "none", "unknown", "n/a", "na"}:
        return np.nan

    texte = texte.replace(",", ".")
    texte = re.sub(r"\babout\b|\bapprox\b|\bapproximately\b|\baround\b|\bover\b|\bunder\b", "", texte)
    texte = texte.replace("half an hour", "0.5 hour")
    texte = texte.replace("half hour", "0.5 hour")
    texte = texte.replace("a few minutes", "3 minutes")

    nombres = re.findall(r"\d+(?:\.\d+)?", texte)
    if not nombres:
        return np.nan

    valeurs = [float(nombre) for nombre in nombres]
    valeur = sum(valeurs) / len(valeurs)

    if re.search(r"day|days", texte):
        return valeur * 86400
    if re.search(r"hour|hours|hr|hrs", texte):
        return valeur * 3600
    if re.search(r"minute|minutes|min|mins", texte):
        return valeur * 60
    if re.search(r"second|seconds|sec|secs", texte):
        return valeur

    return np.nan


## 6. Extraction d'une durée depuis le texte

In [6]:
df["duration_from_text_seconds"] = df["duration_hours_min_clean"].apply(
    parse_duration_to_seconds
)

df[[
    "duration_hours_min",
    "duration_seconds",
    "duration_from_text_seconds",
]].head(20)


,duration_hours_min,duration_seconds,duration_from_text_seconds
0,45 minutes,2700.0,2700.0
1,1-2 hrs,7200.0,5400.0
2,20 seconds,20.0,20.0
3,1/2 hour,20.0,5400.0
4,15 minutes,900.0,900.0
5,5 minutes,300.0,300.0
6,about 3 mins,180.0,180.0
7,20 minutes,1200.0,1200.0
8,3 minutes,180.0,180.0
9,several minutes,120.0,NaN


## 7. Définition des contradictions

Une contradiction est enregistrée lorsque les deux colonnes fournissent une durée positive mais qu'elles diffèrent fortement. Le seuil retenu est un rapport supérieur ou égal à 2 entre les deux durées. Les cas où `duration_seconds` vaut 0 alors que le texte fournit une durée positive sont également considérés comme contradictoires.

In [7]:
duree_numerique_positive = df["duration_seconds"].notna() & (df["duration_seconds"] > 0)
duree_texte_positive = df["duration_from_text_seconds"].notna() & (df["duration_from_text_seconds"] > 0)

rapport_durees = (
    df["duration_seconds"]
    / df["duration_from_text_seconds"]
)

contradiction_forte = (
    duree_numerique_positive
    & duree_texte_positive
    & ((rapport_durees >= 2) | (rapport_durees <= 0.5))
)

zero_contre_texte = (
    (df["duration_seconds"] == 0)
    & duree_texte_positive
)

df["durees_contradictoires"] = (
    contradiction_forte | zero_contre_texte
)

print(f"Contradictions détectées : {int(df['durees_contradictoires'].sum())}")


Contradictions détectées : 1398


## 8. Construction de la durée finale

Règle retenue :

- une durée numérique strictement positive est utilisée lorsqu'elle n'est pas contradictoire ;
- une durée extraite du texte est utilisée si la durée numérique est absente, nulle, négative ou contradictoire ;
- sinon, la durée finale reste manquante.

Aucune ligne n'est supprimée.

In [8]:
duree_numerique_valide = df["duration_seconds"].notna() & (df["duration_seconds"] > 0)

df["duration_finale_seconds"] = np.where(
    duree_numerique_valide & ~df["durees_contradictoires"],
    df["duration_seconds"],
    df["duration_from_text_seconds"],
)

df["source_duree_finale"] = np.select(
    [
        duree_numerique_valide & ~df["durees_contradictoires"],
        df["duration_from_text_seconds"].notna(),
    ],
    [
        "duration_seconds",
        "duration_hours_min_parsee",
    ],
    default="inutilisable",
)

assert len(df) == len(lignes_valides)


## 9. Statistiques demandées

In [9]:
nombre_durees_inutilisables = int(df["duration_finale_seconds"].isna().sum())
nombre_contradictions = int(df["durees_contradictoires"].sum())
duree_mediane = df["duration_finale_seconds"].median()
nombre_plus_une_journee = int((df["duration_finale_seconds"] > 86400).sum())

resume_durees = pd.DataFrame([
    {
        "nombre_lignes_avant": len(lignes_valides),
        "nombre_lignes_apres": len(df),
        "durees_inutilisables": nombre_durees_inutilisables,
        "durees_contradictoires": nombre_contradictions,
        "duree_mediane_secondes": duree_mediane,
        "duree_mediane_minutes": duree_mediane / 60 if pd.notna(duree_mediane) else np.nan,
        "durees_plus_une_journee": nombre_plus_une_journee,
    }
])

resume_durees


,nombre_lignes_avant,nombre_lignes_apres,durees_inutilisables,durees_contradictoires,duree_mediane_secondes,duree_mediane_minutes,durees_plus_une_journee
0,88679,88679,7019,1398,180.0,3.0,222


## 10. Analyse des natures d'aberrations

Deux types d'aberrations sont comptés : les contradictions entre les deux colonnes et les durées supérieures à une journée.

In [10]:
aberrations_durees = pd.DataFrame([
    {
        "type_aberration": "Deux colonnes de durée contradictoires",
        "nombre": nombre_contradictions,
    },
    {
        "type_aberration": "Durée finale supérieure à une journée",
        "nombre": nombre_plus_une_journee,
    },
    {
        "type_aberration": "Durée finale inutilisable",
        "nombre": nombre_durees_inutilisables,
    },
])

aberrations_durees


,type_aberration,nombre
0,Deux colonnes de durée contradictoires,1398
1,Durée finale supérieure à une journée,222
2,Durée finale inutilisable,7019


## 11. Exemples de contradictions

In [11]:
exemples_contradictions = df.loc[
    df["durees_contradictoires"],
    [
        "datetime", "city", "duration_seconds",
        "duration_hours_min", "duration_from_text_seconds",
        "duration_finale_seconds", "source_duree_finale",
    ],
]

exemples_contradictions.head(20)


,datetime,city,duration_seconds,duration_hours_min,duration_from_text_seconds,duration_finale_seconds,source_duree_finale
3,10/10/1956 21:00,edna,20.0,1/2 hour,5400.0,5400.0,duration_hours_min_parsee
50,10/10/1990 20:00,pense (canada),180.0,1/2 hr.,5400.0,5400.0,duration_hours_min_parsee
56,10/10/1992 20:15,seymour,60.0,1min. 39s,1200.0,1200.0,duration_hours_min_parsee
134,10/10/2003 21:10,crescent beach,37800.0,1 1/2 hr.,4800.0,4800.0,duration_hours_min_parsee
150,10/10/2004 21:00,faribault,900.0,1/2 hour,5400.0,5400.0,duration_hours_min_parsee
310,10/11/1989 02:30,copenhagen (denmark),180.0,3min.max&#39,1260.0,1260.0,duration_hours_min_parsee
572,10/1/1973 21:00,roseburg,7200.0,not sure&#44maybe 2 hours,82800.0,82800.0,duration_hours_min_parsee
574,10/1/1973 21:30,hixson,37800.0,1 1/2 hours,4800.0,4800.0,duration_hours_min_parsee
620,10/1/1985 21:30,santa margarita/atascadero (between&#44 hwy 101),120.0,2 &#33/2 min.,740.0,740.0,duration_hours_min_parsee
730,10/1/2000 01:00,vaughn,540.0,9:00 minutes,270.0,270.0,duration_hours_min_parsee


## 12. Trois durées finales les plus longues

Les durées extrêmement élevées sont affichées mais ne sont pas supprimées. Elles sont conservées dans le fichier et dans la colonne finale, car la décision retenue est de ne pas imposer de plafond arbitraire sans connaissance métier démontrant qu'une observation longue est impossible.

In [12]:
trois_durees_plus_longues = df.loc[
    df["duration_finale_seconds"].notna(),
    [
        "datetime", "city", "country", "shape",
        "duration_seconds", "duration_hours_min",
        "duration_from_text_seconds", "duration_finale_seconds",
        "source_duree_finale", "comments",
    ],
]

trois_durees_plus_longues = trois_durees_plus_longues.nlargest(
    3,
    "duration_finale_seconds",
)

trois_durees_plus_longues


,datetime,city,country,shape,duration_seconds,duration_hours_min,duration_from_text_seconds,duration_finale_seconds,source_duree_finale,comments
39963,4/1/2002 22:00,sneads ferry,us,rectangle,0.0,saturday april 27&#442002,1.909565e+10,1.909565e+10,duration_hours_min_parsee,Hello its me agin&#44the person from Sneads Fe...
609,10/1/1983 17:00,birmingham (uk/england),gb,sphere,97836000.0,31 years,NaN,9.783600e+07,duration_seconds,Firstly&#44 I was stunned and stared at the ob...
59230,6/3/2010 23:30,ottawa (canada),ca,other,82800000.0,23000hrs,8.280000e+07,8.280000e+07,duration_seconds,((HOAX??)) I was out in a field near mil&#44 ...


## 13. Export des résultats

In [13]:
resume_durees.to_csv(
    PHASE11_DIR / "resume_durees.csv",
    index=False,
)

aberrations_durees.to_csv(
    PHASE11_DIR / "aberrations_durees.csv",
    index=False,
)

exemples_contradictions.to_csv(
    PHASE11_DIR / "exemples_durees_contradictoires.csv",
    index=False,
)

trois_durees_plus_longues.to_csv(
    PHASE11_DIR / "trois_durees_plus_longues.csv",
    index=False,
)

df[[
    "duration_seconds", "duration_hours_min",
    "duration_from_text_seconds", "duration_finale_seconds",
    "source_duree_finale", "durees_contradictoires",
]].to_csv(
    PHASE11_DIR / "durees_traitees.csv",
    index=False,
)

print(PHASE11_DIR)


..\outputs\phase_11_durees
